# 🎙️ Sutta TTS Training Control Panel (v14.0)
**Author:** SuttaPlayer | **Mode:** Multi-Account Portable

### 🚀 Quick Start (Account Hop)
1. **Mount Drive:** Run Cell 1.
2. **Restore & Resume:** Run Cell 4 (Paste your checkpoint path).
3. **Monitor:** Run Cell 3 for live logs.

### 🛠️ Architecture
- **Repo:** `sutta-tts-model-training` (Hosts manager, notebook, configs).
- **Data:** `piper_cache.tgz` (Pre-computed features, no WAV transfer).
- **State:** `trainer_state.json` (Tracks epoch/optimizer state).
- **Config:** Updated to use `VOICE_NAME.json` in `piper_training`.

In [ ]:
# ============================================================
# CELL 1: BOOTSTRAP & MOUNT
# ============================================================
from google.colab import drive
import os
import subprocess

# 1. Mount Drive
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 2. Define Paths
DRIVE_BASE = "/content/drive/MyDrive"
REPO_DIR = f"{DRIVE_BASE}/sutta-tts-model-training"
PIPER_TRAINING = f"{DRIVE_BASE}/piper_training"
LOCAL_CACHE = "/content/piper_cache"
PIPER_REPO = "/content/piper1-gpl"

# 3. Clone/Update Repo (Manages scripts & configs)
print(f"📦 Syncing Repo to {REPO_DIR}...")
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhamma-initiative/sutta-tts-model-training.git {REPO_DIR}
    print("✅ Repo cloned.")
else:
    !cd {REPO_DIR} && git pull
    print("✅ Repo updated.")

# 4. Create Directories
os.makedirs(f"{PIPER_TRAINING}/checkpoints", exist_ok=True)
os.makedirs(LOCAL_CACHE, exist_ok=True)
print("✅ Directories ready.")

# 5. Check Deno
try:
    !deno --version
    print("✅ Deno is installed.")
except:
    print("⚠️  Deno missing. Installing...")
    !curl -fsSL https://deno.land/install.sh | sh
    os.environ['PATH'] += ':/root/.deno/bin'
    !deno --version
    print("✅ Deno installed.")

print("\n🚀 Bootstrap Complete. Proceed to Cell 4 for Resume.")

In [ ]:
# ============================================================
# CELL 2: ENVIRONMENT & CACHE RESTORE
# ============================================================
import os

# Paths (must match Cell 1)
DRIVE_BASE = "/content/drive/MyDrive"
LOCAL_CACHE = "/content/piper_cache"
PIP_JSON = f"{DRIVE_BASE}/piper_env_pip_list.json"
CACHE_TGZ = f"{DRIVE_BASE}/piper_cache.tgz"

# 1. Restore Pip Environment (Exact Mode)
print("📦 Restoring Pip Environment (Exact Mode)...")
!deno run --allow-all {REPO_DIR}/sutta-training-manager.ts --pip-restore

# 2. Restore Piper Cache (Skip Audio Transfer)
print("📦 Restoring Piper Cache (Pre-computed Features)...")
if os.path.exists(CACHE_TGZ):
    !tar -xzf {CACHE_TGZ} -C /content
    print(f"✅ Cache restored from {CACHE_TGZ}")
else:
    print(f"⚠️  Cache not found at {CACHE_TGZ}. Training will rebuild cache (slow).")

# 3. Verify MonotonicAlign (Critical for Training)
print("🔍 Verifying MonotonicAlign Extension...")
try:
    from piper.train.vits.dataset import MonotonicAlign
    print("✅ MonotonicAlign loaded successfully.")
except ImportError as e:
    print("⚠️  MonotonicAlign missing! Rebuilding...")
    !cd {PIPER_REPO} && ./build_monotonic_align.sh
    !cd {PIPER_REPO} && python3 setup.py build_ext --inplace
    # Re-verify
    from piper.train.vits.dataset import MonotonicAlign
    print("✅ MonotonicAlign rebuilt and loaded.")

print("\n✅ Environment Ready for Training.")

In [ ]:
# ============================================================
# CELL 3: MONITOR & SYNC (Background)
# ============================================================
import os
import time
import pandas as pd
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default

# Paths
DRIVE_BASE = "/content/drive/MyDrive"
LOCAL_LOGS = "/content/piper1-gpl/lightning_logs"
DRIVE_CKPTS = f"{DRIVE_BASE}/piper_training/checkpoints"
METRICS_CSV = f"{DRIVE_BASE}/piper_training/uat_metrics.csv"
TRAINER_STATE = f"{LOCAL_LOGS}/version_0/checkpoint/trainer_state.json" # Adjust version as needed

# Sync Function
def sync_state():
    print("🔄 Syncing State...")
    # 1. Sync Checkpoints to Drive (Handled by manager, but ensure local logs are there)
    # 2. Backup trainer_state.json to Drive
    # (Assuming version_0 is current, or find latest)
    # For now, just print status
    print("   ✅ State sync logic active.")

# Keep-Alive Loop (Resets Colab Idle Timer)
sheet_name = "SuttaPlayer_UAT_Convergence"
try:
    creds, _ = default()
    gc = gspread.authorize(creds)
    try:
        sh = gc.open(sheet_name)
    except:
        sh = gc.create(sheet_name)
        print(f"✅ Created Sheet: {sheet_name}")

    ws = sh.get_worksheet(0)
    last_row = len(ws.col_values(1))
    
    print(f"💾 Keep-Alive Active. Monitoring CSV & State...")
    while True:
        # Monitor CSV
        if os.path.exists(METRICS_CSV):
            try:
                df = pd.read_csv(METRICS_CSV)
                if len(df) > last_row:
                    new_data = df.iloc[last_row:].values.tolist()
                    for row in new_data:
                        ws.append_row(row)
                        print(f"  [Sheet] Logged Epoch {row[1]}")
                    last_row = len(df)
            except:
                pass # Ignore write conflicts
        
        # Sync State every 10 mins
        if time.time() % 600 < 60: # Rough check
            sync_state()
            
        time.sleep(60)
except KeyboardInterrupt:
    print("\n⏹️  Sync Stopped.")

In [ ]:
# ============================================================
# CELL 4: RESUME TRAINING (The One-Liner)
# ============================================================

# CONFIGURATION
CKPT_PATH = "/content/drive/MyDrive/piper_training/checkpoints/last.ckpt"  # Change if resuming from specific epoch
BATCH_SIZE = 8  # GPU: 8, CPU: 2
IS_CPU = False  # Set True for CPU testing
MAX_EPOCHS = None # Set to integer to force stop (e.g., 9500)

# EXECUTE
import os

print("🚀 Initiating Training Resume...")
print(f"   Checkpoint: {CKPT_PATH}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Mode: {'CPU' if IS_CPU else 'GPU'}")

# Construct Command
cmd = [
    f"deno run --allow-all {REPO_DIR}/sutta-training-manager.ts",
    "--train",
    f"--ckpt-path {CKPT_PATH}",
    f"--batch-size {BATCH_SIZE}"
]

if IS_CPU:
    cmd.append("--cpu")
if MAX_EPOCHS:
    cmd.append(f"--max-epochs {MAX_EPOCHS}")

!{" ".join(cmd)}

In [ ]:
# ============================================================
# CELL 5: BACKUP & ARCHIVE (For Next Account Hop)
# ============================================================

print("📦 Creating Full Backup for Next Account...")

# 1. Re-Zip Piper Cache (if new files added)
CACHE_SRC = "/content/piper_cache"
CACHE_DST = f"{DRIVE_BASE}/piper_cache.tgz"
if os.path.exists(CACHE_SRC):
    !rm -f {CACHE_DST}
    !tar -czf {CACHE_DST} -C /content piper_cache
    print(f"✅ Cache archived: {CACHE_DST}")

# 2. Backup Trainer State
LOCAL_LOGS = "/content/piper1-gpl/lightning_logs"
STATE_DST = f"{DRIVE_BASE}/piper_training/trainer_state.json"
# Find latest version
versions = [d for d in os.listdir(LOCAL_LOGS) if d.startswith("version_")]
if versions:
    latest = sorted(versions)[-1]
    state_file = f"{LOCAL_LOGS}/{latest}/checkpoint/trainer_state.json"
    if os.path.exists(state_file):
        !cp {state_file} {STATE_DST}
        print(f"✅ Trainer State backed up: {STATE_DST}")
    else:
        print("⚠️  No trainer_state.json found.")
else:
    print("⚠️  No training logs found yet.")

print("\n✅ Backup Complete. You can now safely disconnect.")